<a href="https://colab.research.google.com/github/youngPath12/AI-Health-Bio-Data-Team-4-Professor-yoon-/blob/Jimin/brain_age_gap_alzheimer_vulnerability_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 기계학습 기반 Brain Age Gap 예측을 통한 알츠하이머 취약성 분석

이 노트북은 ADNI 구조적 T1 MRI의 FreeSurfer ROI를 이용해 **정상 인지군(NCI)의 달력 나이**를 예측하고, 실제 나이와의 차이인 **Brain Age Gap (BAG)** 으로 MCI 및 알츠하이머병(AD)의 취약성을 평가합니다.

핵심 원칙은 질병으로 인한 뇌 위축을 정상 노화로 학습하지 않도록, **나이 예측 모델은 NCI만으로 훈련**한다는 점입니다. BAG가 양수이면 뇌가 실제 나이보다 더 늙은 방향, 음수이면 더 젊은 방향으로 예측된 것입니다. 이 결과는 연구용 연관성 지표이며 개인 진단이나 치료 결정을 위한 도구가 아닙니다.

분석 순서: 데이터 로드 → 기저시점 1회 선택 → NCI 기반 나이모델 → 편향 보정 BAG → 진단군/인지점수 연관 분석 → 해석 가능한 중요 ROI 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. Colab 데이터 준비

가장 편한 방법은 현재 `program_data` 폴더 전체를 `program_data.zip`으로 압축해 Google Drive에 올리는 것입니다. 아래 셀에서 `USE_GOOGLE_DRIVE=True`로 두고 `ZIP_PATH`만 수정하세요. 압축을 원하지 않으면 Drive에 폴더를 그대로 올린 뒤 `DATA_ROOT`를 그 폴더 경로로 지정하고 `USE_ZIP=False`로 바꿉니다.

필수 파일은 `ADSP_PHC/ADSP_PHC_T1_FS_22Jan2026.csv`입니다. 인지기능 분석에는 `Assessments/MMSE_22Jan2026.csv`, `Assessments/ADAS_22Jan2026.csv`도 사용합니다.

In [ ]:
# Colab 기본 라이브러리. 처음 한 번만 실행하면 됩니다.
!pip -q install seaborn statsmodels

from pathlib import Path
from zipfile import ZipFile
from google.colab import drive

USE_GOOGLE_DRIVE = True
USE_ZIP = True
ZIP_PATH = '/content/drive/MyDrive/program_data/ADNI_data_Do_NOT_redistribute (1).zip'  # 본인의 Drive 경로로 수정
DATA_ROOT = Path('/content/drive/MyDrive/program_data')             # USE_ZIP=False일 때의 폴더 경로

if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')

if USE_ZIP:
    extract_root = Path('/content/adni_project_data')
    extract_root.mkdir(exist_ok=True)
    with ZipFile(ZIP_PATH) as zf:
        zf.extractall(extract_root)

    # Check for ADSP_PHC.zip and extract it if found
    adsp_phc_zip_path = extract_root / 'ADSP_PHC.zip'
    if adsp_phc_zip_path.exists():
        adsp_phc_extract_dir = extract_root / 'ADSP_PHC_extracted' # Create a new directory for ADSP_PHC.zip contents
        adsp_phc_extract_dir.mkdir(exist_ok=True)
        with ZipFile(adsp_phc_zip_path) as zf_adsp:
            zf_adsp.extractall(adsp_phc_extract_dir)
        print(f"Extracted {adsp_phc_zip_path} to {adsp_phc_extract_dir}")
        # Now search for the CSV file within this newly extracted directory
        t1_candidates = list(adsp_phc_extract_dir.rglob('ADSP_PHC_T1_FS_22Jan2026.csv'))
    else:
        # If ADSP_PHC.zip is not found, search in the main extract_root directly
        t1_candidates = list(extract_root.rglob('ADSP_PHC_T1_FS_22Jan2026.csv'))

    if len(t1_candidates) != 1:
        raise FileNotFoundError(f'T1 FreeSurfer 파일을 하나만 찾아야 합니다. 발견 수: {len(t1_candidates)}')
    # The DATA_ROOT should point to the base directory where other shared data (like 'Assessments') is found.
    # In this case, it's the `extract_root`.
    DATA_ROOT = extract_root

print('DATA_ROOT:', DATA_ROOT)
# The previous assertion expected a specific subdirectory structure that doesn't match the current extraction.
# The `t1_candidates[0]` now holds the correct direct path to the file. We will use this directly.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracted /content/adni_project_data/ADSP_PHC.zip to /content/adni_project_data/ADSP_PHC_extracted
DATA_ROOT: /content/adni_project_data


In [ ]:
print(f"Listing contents of the extracted directory: {extract_root}")
for path in extract_root.rglob('*'):
    print(path)

Listing contents of the extracted directory: /content/adni_project_data
/content/adni_project_data/Quick_Start.zip
/content/adni_project_data/Test_Data_for_Challenges_except_imaging_vertices.zip
/content/adni_project_data/Curated_Data___Docs.zip
/content/adni_project_data/Study_Info.zip
/content/adni_project_data/Genetic.zip
/content/adni_project_data/Remotely_Collected_Data.zip
/content/adni_project_data/desktop.ini
/content/adni_project_data/Assessments.zip
/content/adni_project_data/Subject_Characteristics.zip
/content/adni_project_data/ADSP_PHC.zip
/content/adni_project_data/Neuropathology_Results.zip
/content/adni_project_data/Medical_History.zip
/content/adni_project_data/Imaging.zip


위의 출력에서 `ADSP_PHC_T1_FS_22Jan2026.csv` 파일의 정확한 경로를 확인해 주세요.

만약 파일이 `ADSP_PHC/ADSP_PHC_T1_FS_22Jan2026.csv`와 같은 경로에 있다면, 현재 `DATA_ROOT = t1_candidates[0].parents[1]` 로직은 올바른 `DATA_ROOT`를 설정해야 합니다.

하지만 `extract_root` 바로 아래에 `ADSP_PHC_T1_FS_22Jan2026.csv` 파일이 있거나 다른 경로에 있다면, `t1_candidates[0].parents[1]` 대신 다른 `parents` 레벨을 사용하거나 `DATA_ROOT`를 직접 설정해야 할 수도 있습니다.

예를 들어, 파일이 `/content/adni_project_data/ADNI_data_Do_NOT_redistribute (1)/ADSP_PHC/ADSP_PHC_T1_FS_22Jan2026.csv`와 같은 경로에 있다면, `ZIP_PATH` 안에 `ADNI_data_Do_NOT_redistribute (1)`이라는 추가적인 상위 폴더가 있는 것입니다. 이 경우, `t1_candidates[0].parents[1]` 대신 `t1_candidates[0].parents[2]`를 사용해야 할 수 있습니다.

## 1. 분석용 데이터 만들기

`PHC_Diagnosis`는 1=NCI, 2=MCI, 3=AD입니다. 같은 사람이 여러 MRI를 갖고 있으므로, 주 분석에서는 각 참가자의 **가장 이른 방문 1회**만 남깁니다. 추적자료까지 동시에 모델에 넣으면 한 사람의 MRI가 학습·평가에 섞이는 누수 위험이 있습니다.

입력 특성은 ComBat 보정이 적용된 `_combat` ROI입니다. 나이, 진단, RID와 같은 결과·식별 변수는 특성에서 제외합니다. 결측값은 이후 파이프라인 안에서 중앙값으로 대치하므로 검증 세트의 정보를 학습에 사용하지 않습니다.

In [ ]:
import numpy as np
import pandas as pd

T1_PATH = t1_candidates[0] # Use the directly found path for the T1 FreeSurfer CSV
t1 = pd.read_csv(T1_PATH, low_memory=False)

# 방문 시점의 정렬 기준: 검사일이 없으면 파일 순서를 보조적으로 사용
t1['PHC_SCANDATE'] = pd.to_datetime(t1['PHC_SCANDATE'], errors='coerce')
t1 = t1.dropna(subset=['RID', 'PHC_Age_T1', 'PHC_Diagnosis']).copy()
t1['PHC_Diagnosis'] = pd.to_numeric(t1['PHC_Diagnosis'], errors='coerce')
t1 = t1[t1['PHC_Diagnosis'].isin([1, 2, 3])].copy()

# 참가자별 가장 이른 구조 MRI 1개만 선택
analysis_df = (t1.sort_values(['RID', 'PHC_SCANDATE', 'VISCODE2'], na_position='last')
                  .drop_duplicates('RID', keep='first')
                  .reset_index(drop=True))

feature_cols = [c for c in analysis_df.columns if c.endswith('_combat')]
# 전부 결측이거나 상수인 열은 제거
feature_cols = [c for c in feature_cols
                if analysis_df[c].notna().sum() >= 0.70 * len(analysis_df)
                and analysis_df[c].nunique(dropna=True) > 1]

diagnosis_label = {1: 'NCI', 2: 'MCI', 3: 'AD'}
analysis_df['DX_GROUP'] = analysis_df['PHC_Diagnosis'].map(diagnosis_label)
print(f'분석 대상: {len(analysis_df):,}명, MRI 특성: {len(feature_cols)}개')
display(analysis_df['DX_GROUP'].value_counts().reindex(['NCI','MCI','AD']).to_frame('n'))
display(analysis_df[['PHC_Age_T1', 'DX_GROUP', 'PHC_Sex', 'PHC_FieldSgth']].describe(include='all'))


분석 대상: 2,385명, MRI 특성: 192개


,n
DX_GROUP,
NCI,911
MCI,1088
AD,386


,PHC_Age_T1,DX_GROUP,PHC_Sex,PHC_FieldSgth
count,2385.000000,2385,2385.000000,2385.000000
unique,NaN,3,NaN,NaN
top,NaN,MCI,NaN,NaN
freq,NaN,1088,NaN,NaN
mean,73.035304,NaN,1.487212,2.591321
std,7.377553,NaN,0.499941,0.667445
min,49.000000,NaN,1.000000,1.500000
25%,68.100000,NaN,1.000000,1.500000
50%,73.000000,NaN,1.000000,3.000000
75%,78.100000,NaN,2.000000,3.000000


## 2. NCI에서 뇌나이 모델 학습 및 성능 평가

NCI 참가자를 참가자 단위로 학습(80%)·검증(20%)으로 나눕니다. Ridge는 많은 상관된 MRI 변수를 안정적으로 다루는 해석 가능한 기준모델이고, Extra Trees는 비선형 관계를 잡는 비교모델입니다. 성능이 더 좋은 모델을 최종 선택합니다.

주의: NCI 홀드아웃 성능은 모델이 ‘정상 노화’를 얼마나 잘 근사하는지의 성능이며, AD 판별 정확도가 아닙니다.

In [ ]:
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
import numpy as np # numpy import 추가

nci = analysis_df[analysis_df['DX_GROUP'].eq('NCI')].copy()
train_idx, test_idx = train_test_split(nci.index, test_size=0.20, random_state=42)
train_nci, test_nci = nci.loc[train_idx], nci.loc[test_idx]

models = {
    'Ridge': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', Ridge(alpha=30))]),
    'ExtraTrees': Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', ExtraTreesRegressor(n_estimators=500, min_samples_leaf=3, max_features=0.7, random_state=42, n_jobs=-1))])
}
results, fitted_models = [], {}
for name, model in models.items():
    model.fit(train_nci[feature_cols], train_nci['PHC_Age_T1'])
    pred = model.predict(test_nci[feature_cols])
    mse = mean_squared_error(test_nci['PHC_Age_T1'], pred) # squared=False 제거
    rmse = np.sqrt(mse) # RMSE 수동 계산
    results.append([name, mean_absolute_error(test_nci['PHC_Age_T1'], pred),
                    rmse, # RMSE 값으로 변경
                    r2_score(test_nci['PHC_Age_T1'], pred)])
    fitted_models[name] = model
performance = pd.DataFrame(results, columns=['model', 'MAE_years', 'RMSE_years', 'R2']).sort_values('MAE_years')
display(performance.style.format({'MAE_years':'{:.2f}', 'RMSE_years':'{:.2f}', 'R2':'{:.3f}'}))
best_name = performance.iloc[0]['model']
print('선택 모델:', best_name)


TypeError: got an unexpected keyword argument 'squared'

## 3. 예측 연령 편향 보정과 BAG 산출

뇌나이 회귀는 평균으로 수축하는 경향이 있어, 저연령은 높게·고연령은 낮게 예측하는 편향이 생깁니다. 이를 막기 위해 훈련 NCI의 5-fold 교차검증 예측값으로 `예측나이 = 절편 + 기울기 × 실제나이`를 추정하고, 모든 예측값에 같은 보정을 적용합니다.

보정 후 `BAG = 보정 예측나이 − 실제나이`입니다. 보정식은 NCI 훈련자료에서만 학습하므로 MCI/AD의 정보를 사용하지 않습니다.

In [ ]:
# 선택 모델을 전체 NCI로 다시 학습하고, OOF 예측으로 편향 보정식을 추정
best_model = clone(models[best_name])
cv = GroupKFold(n_splits=5)
oof_pred = cross_val_predict(best_model, nci[feature_cols], nci['PHC_Age_T1'],
                             groups=nci['RID'], cv=cv, n_jobs=-1)
bias_slope, bias_intercept = np.polyfit(nci['PHC_Age_T1'], oof_pred, 1)
if bias_slope <= 0:
    raise ValueError('편향 보정 기울기가 0 이하입니다. 데이터 및 모델 설정을 점검하세요.')

best_model.fit(nci[feature_cols], nci['PHC_Age_T1'])
analysis_df['predicted_brain_age_raw'] = best_model.predict(analysis_df[feature_cols])
analysis_df['predicted_brain_age'] = (analysis_df['predicted_brain_age_raw'] - bias_intercept) / bias_slope
analysis_df['BAG'] = analysis_df['predicted_brain_age'] - analysis_df['PHC_Age_T1']
print(f'편향 보정식: raw prediction = {bias_intercept:.2f} + {bias_slope:.3f} × chronological age')
display(analysis_df.groupby('DX_GROUP')['BAG'].agg(['count','mean','std','median']).reindex(['NCI','MCI','AD']).round(2))

## 4. 시각화와 진단군 차이 검정

아래 회귀모형은 BAG의 집단 차이를 나이와 성별을 보정해 검정합니다. `C(DX_GROUP)`의 MCI·AD 계수는 기준군 NCI 대비 보정된 BAG 차이(년)입니다. 관찰연구이므로 이 차이를 질병의 원인효과로 해석해서는 안 됩니다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
sns.set_theme(style='whitegrid', context='notebook')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=analysis_df, x='PHC_Age_T1', y='predicted_brain_age', hue='DX_GROUP', alpha=.65, ax=axes[0])
lims = [analysis_df['PHC_Age_T1'].min()-2, analysis_df['PHC_Age_T1'].max()+2]
axes[0].plot(lims, lims, 'k--', lw=1); axes[0].set(xlim=lims, ylim=lims, xlabel='Chronological age', ylabel='Predicted brain age')
sns.boxplot(data=analysis_df, x='DX_GROUP', y='BAG', order=['NCI','MCI','AD'], ax=axes[1], showfliers=False)
sns.stripplot(data=analysis_df, x='DX_GROUP', y='BAG', order=['NCI','MCI','AD'], ax=axes[1], color='black', alpha=.22, size=2)
axes[1].axhline(0, color='black', ls='--', lw=1); axes[1].set(xlabel='', ylabel='Brain Age Gap (years)')
plt.tight_layout(); plt.show()

group_model = smf.ols('BAG ~ C(DX_GROUP, Treatment(reference="NCI")) + PHC_Age_T1 + C(PHC_Sex)', data=analysis_df).fit(cov_type='HC3')
print(group_model.summary())

## 5. 인지기능과의 수렴 타당성

MMSE는 높을수록 인지기능이 양호하고, ADAS13은 높을수록 장애가 심합니다. 따라서 유효한 BAG라면 (전체 및 진단군 보정 후) 일반적으로 MMSE와 음의 관계, ADAS13과 양의 관계가 기대됩니다.

임상평가와 MRI 방문코드가 정확히 일치하지 않는 경우가 있어, 이 셀은 동일 `RID`와 `VISCODE2`만 연결합니다. 연결 수가 적다면 가장 가까운 검사일을 기준으로 매칭하는 확장 분석을 별도로 수행하세요.

In [ ]:
MMSE_PATH = DATA_ROOT / 'Assessments' / 'MMSE_22Jan2026.csv'
ADAS_PATH = DATA_ROOT / 'Assessments' / 'ADAS_22Jan2026.csv'
mmse = pd.read_csv(MMSE_PATH, low_memory=False)[['RID','VISCODE2','MMSCORE']].drop_duplicates(['RID','VISCODE2'])
adas = pd.read_csv(ADAS_PATH, low_memory=False)[['RID','VISCODE2','TOTAL13']].drop_duplicates(['RID','VISCODE2'])
clinical = analysis_df.merge(mmse, on=['RID','VISCODE2'], how='left').merge(adas, on=['RID','VISCODE2'], how='left')
print('MMSE 연결:', clinical['MMSCORE'].notna().sum(), '명 | ADAS13 연결:', clinical['TOTAL13'].notna().sum(), '명')

for outcome in ['MMSCORE', 'TOTAL13']:
    tmp = clinical.dropna(subset=[outcome]).copy()
    model = smf.ols(f'{outcome} ~ BAG + C(DX_GROUP) + PHC_Age_T1 + C(PHC_Sex)', data=tmp).fit(cov_type='HC3')
    print(f'\n[{outcome}] BAG 계수와 유의성 (진단군·나이·성별 보정)')
    print(model.summary().tables[1])
    sns.lmplot(data=tmp, x='BAG', y=outcome, hue='DX_GROUP', height=4.5, aspect=1.3, scatter_kws={'alpha':.45, 's':20})
    plt.show()

## 6. 모델 해석과 결과 저장

Extra Trees가 선택된 경우 permutation importance는 예측 성능에 특히 기여한 ROI를 보여 줍니다. 이는 인과적 뇌 영역 목록이 아니라 모델 의존적 중요도입니다. Ridge가 선택되면 표준화 계수의 절대값을 사용합니다. 마지막 셀은 이후 재현 및 보고서 작성에 필요한 개인별 BAG 결과를 CSV로 저장합니다.

In [ ]:
from sklearn.inspection import permutation_importance

if best_name == 'ExtraTrees':
    pi = permutation_importance(best_model, test_nci[feature_cols], test_nci['PHC_Age_T1'], n_repeats=10, random_state=42, n_jobs=-1, scoring='neg_mean_absolute_error')
    importance = pd.Series(pi.importances_mean, index=feature_cols).sort_values(ascending=False)
else:
    importance = pd.Series(np.abs(best_model.named_steps['model'].coef_), index=feature_cols).sort_values(ascending=False)

top_roi = importance.head(15).sort_values()
plt.figure(figsize=(9, 6)); top_roi.plot.barh(); plt.xlabel('Importance'); plt.title(f'Top MRI features: {best_name}'); plt.show()

output_cols = ['RID','PTID','VISCODE2','PHC_SCANDATE','PHC_Age_T1','DX_GROUP','PHC_Sex','PHC_FieldSgth','predicted_brain_age_raw','predicted_brain_age','BAG']
analysis_df[output_cols].to_csv('brain_age_gap_results.csv', index=False)
performance.to_csv('brain_age_model_performance.csv', index=False)
print('저장 완료: brain_age_gap_results.csv, brain_age_model_performance.csv')